# Sentinel Gate — Stage 0: inspect EMBER2024 Win64 + Win32 (Colab or Kaggle)

**What this does**
1. Checks free disk space and picks the largest scratch disk.
2. Loads the Sentinel Gate code bundle (`sentinel_gate_code.zip`, code only, no data, no secrets).
3. Installs the *locked* dependencies (`uv.lock`) into an isolated Python 3.11 venv.
4. For each file type (Win64, then Win32): downloads the official feature-record zips from HuggingFace `joyce8/EMBER2024` (SHA-256 verified), measures them with `scripts/inspect_dataset.py`, then **deletes the zips** to free disk.
5. Packs only the small results (`dataset_stats/*.json` + a partial report) into `sentinel_stage0_results.zip` for you to download.

**Safety:** EMBER2024 contains *feature records* (JSON numbers/strings) only. No binaries are downloaded or executed. The `EMBER2024-capa` dataset (raw function bytes) is refused by the downloader. Results contain counts only: no file hashes leave this notebook.

**Before running on Kaggle:** Settings → *Internet* must be **On**, and attach the code bundle as a Dataset.

Run all cells top to bottom (Runtime → Run all).

In [ ]:
# 1) Platform + disk check
import os, shutil, subprocess, sys, zipfile, hashlib, glob, time
from pathlib import Path

FILE_TYPES = ["Win64", "Win32"]   # processed one at a time, zips deleted after each
REQUIRED_FREE_GB = 22             # largest type (Win32 ~14.9 GB zips) + venv + cache + margin

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules or Path("/content").exists() and not ON_KAGGLE
WHERE = "Kaggle notebook" if ON_KAGGLE else ("Google Colab" if ON_COLAB else "other notebook host")

candidates = [Path(p) for p in ["/kaggle/tmp", "/kaggle/working", "/content", "/tmp"] if Path(p).exists()]
free = {p: shutil.disk_usage(p).free / 1e9 for p in candidates}
for p, gb in free.items():
    print(f"{p}: {gb:.1f} GB free")
SCRATCH = max(free, key=free.get)
OUT_DIR = Path("/kaggle/working/sentinel_results") if ON_KAGGLE else SCRATCH / "sentinel_results"
print(f"\nHost: {WHERE} | scratch: {SCRATCH} ({free[SCRATCH]:.1f} GB free) | outputs: {OUT_DIR}")
print(f"CPUs: {os.cpu_count()} | RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.1f} GB")
if free[SCRATCH] < REQUIRED_FREE_GB:
    raise SystemExit(f"Not enough disk: need {REQUIRED_FREE_GB} GB free, have {free[SCRATCH]:.1f} GB. Stop here and tell Claude.")

CODE = SCRATCH / "sentinel_code"
DATA = SCRATCH / "sentinel_data"
VENV = SCRATCH / "sentinel_venv"
for d in (DATA / "raw", OUT_DIR / "dataset_stats"):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2) Load the code bundle (Colab: upload it; Kaggle: attach it as a Dataset)
BUNDLE_NAME = "sentinel_gate_code.zip"

def find_bundle():
    zips = glob.glob(f"/kaggle/input/**/{BUNDLE_NAME}", recursive=True) + glob.glob(f"/content/{BUNDLE_NAME}")
    if zips:
        return Path(zips[0]), None
    # Kaggle auto-extracts uploaded zips: look for an extracted copy instead
    dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/BUNDLE_INFO.json", recursive=True)]
    return None, (dirs[0] if dirs else None)

bundle, extracted = find_bundle()
if bundle is None and extracted is None and ON_COLAB:
    from google.colab import files
    print(f"Choose {BUNDLE_NAME} from your computer (dist/{BUNDLE_NAME} in the project):")
    uploaded = files.upload()
    bundle, extracted = find_bundle()
if bundle is None and extracted is None:
    raise SystemExit(f"{BUNDLE_NAME} not found. Colab: upload it. Kaggle: Add Data -> your uploaded dataset.")

shutil.rmtree(CODE, ignore_errors=True)
if bundle is not None:
    print(f"bundle: {bundle}\nsha256: {hashlib.sha256(bundle.read_bytes()).hexdigest()}  <- compare with `make bundle` output")
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(CODE)
else:
    print(f"using extracted bundle at {extracted}")
    shutil.copytree(extracted, CODE)
print(open(CODE / "BUNDLE_INFO.json").read()[:300], "...")

In [ ]:
# 3) Install locked dependencies into an isolated Python 3.11 venv (same uv.lock as the local sandbox)
def run(cmd, env=None, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), env=env, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise SystemExit(f"command failed with exit code {proc.returncode}")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.13.6"])
env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(VENV), UV_PYTHON="3.11", UV_LINK_MODE="copy")
run([sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--no-install-project"], env=env, cwd=CODE)
PY = VENV / "bin" / "python"
RUN_ENV = dict(os.environ, PYTHONPATH=f"{CODE / 'src'}:{CODE}", SENTINEL_DATA_DIR=str(DATA),
               HF_HOME=str(DATA / ".hf-home"), PYTHONUNBUFFERED="1")
run([PY, "-c", "import sys, thrember; print(sys.version.split()[0], 'thrember dim', thrember.PEFeatureExtractor().dim)"], env=RUN_ENV)

In [ ]:
# 4) Download -> inspect -> delete zips, one file type at a time
for ft in FILE_TYPES:
    t0 = time.time()
    run([PY, "scripts/download_ember.py", "--file-types", ft, "--dest", DATA / "raw"], env=RUN_ENV, cwd=CODE)
    run([PY, "scripts/inspect_dataset.py", "--file-types", ft, "--raw-dir", DATA / "raw",
         "--stats-dir", OUT_DIR / "dataset_stats", "--out", OUT_DIR / "DATASET_REPORT_partial.md",
         "--where", WHERE, "--workers", str(os.cpu_count() or 2)], env=RUN_ENV, cwd=CODE)
    for z in (DATA / "raw").glob(f"{ft}_*.zip"):
        z.unlink()
    shutil.rmtree(DATA / ".hf-home", ignore_errors=True)
    print(f"== {ft} done in {(time.time() - t0) / 60:.1f} min; zips deleted; "
          f"{shutil.disk_usage(SCRATCH).free / 1e9:.1f} GB free\n")
shutil.copy(DATA / "raw" / "download_manifest.json", OUT_DIR / "download_manifest.json")

In [ ]:
# 5) Package the small results and hand them back
result_zip = shutil.make_archive(str(OUT_DIR.parent / "sentinel_stage0_results"), "zip", OUT_DIR)
print("results:", result_zip)
with zipfile.ZipFile(result_zip) as zf:
    for i in zf.infolist():
        print(f"  {i.filename}  {i.file_size:,} bytes")
print(open(OUT_DIR / "DATASET_REPORT_partial.md").read()[:3000])
if ON_COLAB:
    from google.colab import files
    files.download(result_zip)
else:
    print("Kaggle: download sentinel_stage0_results.zip from the notebook's Output panel (/kaggle/working).")